# Lab 3: Skip Connections / Residual Blocks

## Topic 3: Flexible ML Architectures with Vibe Coding

In this lab, we explore **skip connections** and **residual blocks** -- the key innovation behind ResNet that enabled training of very deep networks. We will build a mini-ResNet from scratch and compare it against a plain network.

### What You Will Learn
- The vanishing gradient problem and why deep networks are hard to train
- How residual connections solve this problem
- Implementing a reusable `residual_block()` helper function
- Building and training a mini-ResNet on CIFAR-10

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

from keras import layers, Model, Input
from keras.utils import plot_model

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 1. Load and Prepare CIFAR-10

In [ ]:
# Load CIFAR-10
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 2. The Vanishing Gradient Problem

When we stack many layers in a deep network, gradients can shrink exponentially as they are backpropagated through the chain rule. This is the **vanishing gradient problem**.

### Why It Happens
- Each layer multiplies the gradient by its local derivative
- If these derivatives are consistently < 1 (common with sigmoid/tanh), the gradient shrinks
- After many layers, the gradient reaching early layers is negligibly small
- Early layers stop learning effectively

### The ResNet Solution
Instead of learning the full mapping `H(x)`, a residual block learns the **residual** `F(x) = H(x) - x`.

The output becomes: `H(x) = F(x) + x`

The `+ x` (skip connection) creates a direct gradient highway:
- Gradients can flow directly through the skip connection
- Even if `F(x)` gradients vanish, the identity gradient of 1 still propagates
- The network can always fall back to the identity mapping if needed

## 3. Implement the Residual Block

We will implement a reusable `residual_block(x, filters)` function that:
1. Applies Conv2D -> BatchNormalization -> ReLU -> Conv2D -> BatchNormalization
2. Adds the input (shortcut) to the output
3. Applies a final ReLU activation

When the number of filters changes, a 1x1 convolution is used to project the shortcut.

In [ ]:
def residual_block(x, filters, strides=1):
    """A residual block with two Conv2D layers and a skip connection.
    
    Args:
        x: Input tensor
        filters: Number of filters for the Conv2D layers
        strides: Stride for the first Conv2D (use 2 for downsampling)
    
    Returns:
        Output tensor with the residual connection applied
    """
    # Save the input for the skip connection
    shortcut = x
    
    # First convolutional layer
    x = layers.Conv2D(filters, (3, 3), strides=strides, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Second convolutional layer
    x = layers.Conv2D(filters, (3, 3), strides=1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    
    # If the number of filters or spatial dimensions changed,
    # project the shortcut with a 1x1 convolution
    if strides != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), strides=strides, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add the skip connection
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    
    return x

print("residual_block() function defined successfully.")

## 4. Build a Mini-ResNet

Architecture:
```
Input -> Conv2D(32) -> BatchNorm -> ReLU
      -> ResBlock(32) -> ResBlock(64, stride=2) -> ResBlock(128, stride=2)
      -> GlobalAveragePooling2D -> Dense(10, softmax)
```

In [ ]:
def build_mini_resnet(input_shape=(32, 32, 3), num_classes=10):
    """Build a mini-ResNet with 3 residual blocks."""
    inputs = Input(shape=input_shape, name="image_input")
    
    # Initial convolution
    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False, name="initial_conv")(inputs)
    x = layers.BatchNormalization(name="initial_bn")(x)
    x = layers.ReLU(name="initial_relu")(x)
    
    # Residual blocks
    x = residual_block(x, filters=32, strides=1)   # 32x32 -> 32x32
    x = residual_block(x, filters=64, strides=2)   # 32x32 -> 16x16
    x = residual_block(x, filters=128, strides=2)  # 16x16 -> 8x8
    
    # Classification head
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="classification")(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="mini_resnet")
    return model

# Build the mini-ResNet
mini_resnet = build_mini_resnet()
mini_resnet.summary()

In [ ]:
# Visualize the mini-ResNet architecture
plot_model(
    mini_resnet,
    show_shapes=True,
    show_layer_names=True,
    to_file="mini_resnet.png",
    dpi=100,
    expand_nested=True
)

## 5. Build a Plain Network (Same Depth, No Skip Connections)

For comparison, we build a plain network with the same number of convolutional layers but **without** skip connections.

In [ ]:
def build_plain_network(input_shape=(32, 32, 3), num_classes=10):
    """Build a plain CNN with same depth as the mini-ResNet but no skip connections."""
    inputs = Input(shape=input_shape, name="image_input")
    
    # Initial convolution (same as mini-ResNet)
    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 1: 32 filters (equivalent depth to first residual block)
    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 2: 64 filters with stride 2 (equivalent to second residual block)
    x = layers.Conv2D(64, (3, 3), strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 3: 128 filters with stride 2 (equivalent to third residual block)
    x = layers.Conv2D(128, (3, 3), strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(128, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Classification head (same as mini-ResNet)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="plain_network")
    return model

plain_net = build_plain_network()
plain_net.summary()

print(f"\nMini-ResNet parameters: {mini_resnet.count_params():,}")
print(f"Plain Network parameters: {plain_net.count_params():,}")

## 6. Train Both Models

In [ ]:
# Compile both models with the same settings
for model in [mini_resnet, plain_net]:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

print("Both models compiled.")

In [ ]:
# Train the mini-ResNet
print("Training Mini-ResNet...")
print("=" * 50)
resnet_history = mini_resnet.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Train the plain network
print("Training Plain Network...")
print("=" * 50)
plain_history = plain_net.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

## 7. Evaluate and Compare

In [ ]:
# Evaluate both models
resnet_loss, resnet_acc = mini_resnet.evaluate(X_test, y_test, verbose=0)
plain_loss, plain_acc = plain_net.evaluate(X_test, y_test, verbose=0)

print(f"Mini-ResNet   - Test Loss: {resnet_loss:.4f}, Test Accuracy: {resnet_acc:.4f}")
print(f"Plain Network - Test Loss: {plain_loss:.4f}, Test Accuracy: {plain_acc:.4f}")

In [ ]:
# Plot training history comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(resnet_history.history["accuracy"], label="ResNet (train)", color="blue")
axes[0].plot(resnet_history.history["val_accuracy"], label="ResNet (val)", color="blue", linestyle="--")
axes[0].plot(plain_history.history["accuracy"], label="Plain (train)", color="red")
axes[0].plot(plain_history.history["val_accuracy"], label="Plain (val)", color="red", linestyle="--")
axes[0].set_title("Accuracy: ResNet vs Plain Network")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(resnet_history.history["loss"], label="ResNet (train)", color="blue")
axes[1].plot(resnet_history.history["val_loss"], label="ResNet (val)", color="blue", linestyle="--")
axes[1].plot(plain_history.history["loss"], label="Plain (train)", color="red")
axes[1].plot(plain_history.history["val_loss"], label="Plain (val)", color="red", linestyle="--")
axes[1].set_title("Loss: ResNet vs Plain Network")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Mini-ResNet vs Plain Network (Same Depth)", fontsize=14)
plt.tight_layout()
plt.show()

# Summary
print("\n" + "="*60)
print(f"{'Model':<20} {'Parameters':<15} {'Test Loss':<12} {'Test Acc':<12}")
print("="*60)
print(f"{'Mini-ResNet':<20} {mini_resnet.count_params():<15,} {resnet_loss:<12.4f} {resnet_acc:<12.4f}")
print(f"{'Plain Network':<20} {plain_net.count_params():<15,} {plain_loss:<12.4f} {plain_acc:<12.4f}")
print("="*60)
print(f"\nAccuracy difference: {(resnet_acc - plain_acc)*100:+.2f}%")

In [ ]:
# Visualize predictions on test samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Pick 10 random test images
indices = np.random.choice(len(X_test), 10, replace=False)

for idx, ax in zip(indices, axes.flat):
    img = X_test[idx]
    true_label = CLASS_NAMES[y_test[idx][0]]
    
    # Get ResNet prediction
    pred = mini_resnet.predict(np.expand_dims(img, 0), verbose=0)[0]
    pred_label = CLASS_NAMES[np.argmax(pred)]
    confidence = np.max(pred) * 100
    
    ax.imshow(img)
    color = "green" if pred_label == true_label else "red"
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.0f}%)",
                 fontsize=9, color=color)
    ax.axis("off")

plt.suptitle("Mini-ResNet Predictions on Test Images", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Gradio Interface

Classify images interactively using the trained mini-ResNet.

In [ ]:
import gradio as gr

def classify_with_resnet(image):
    """Classify an uploaded image using the mini-ResNet."""
    if image is None:
        return {name: 0.0 for name in CLASS_NAMES}
    
    # Preprocess: resize to 32x32 and normalize
    import PIL.Image
    img = PIL.Image.fromarray(image).resize((32, 32))
    img_array = np.array(img).astype("float32") / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    predictions = mini_resnet.predict(img_array, verbose=0)[0]
    
    return {CLASS_NAMES[i]: float(predictions[i]) for i in range(10)}

# Create Gradio interface
demo = gr.Interface(
    fn=classify_with_resnet,
    inputs=gr.Image(label="Upload an Image"),
    outputs=gr.Label(num_top_classes=5, label="Predictions"),
    title="CIFAR-10 Classifier (Mini-ResNet with Skip Connections)",
    description=(
        "Upload an image to classify it using a mini-ResNet architecture with residual blocks. "
        "The model uses skip connections to enable better gradient flow and improved training. "
        "Categories: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck."
    ),
    flagging_mode="never"
)

demo.launch()